# 🛰️ Distributed Satellite Image Processing for Coastal Change Detection
## Using Apache PySpark | Study Site: Puthuvype Beach, Kerala, India

| Item | Detail |
|------|--------|
| **Author** | Akhila K A |
| **Role** | Big Data Lead Engineer — Interview Demo |
| **Framework** | Apache PySpark 3.5.x |
| **Use Case** | Distributed coastal change detection (2018 vs 2020) |

### Pipeline Architecture
```
Upload Images → Resize (512×512) → Tile (64×64) → Feature Extraction
      → Spark DataFrame → Distributed Join → Change Detection
      → CSV Export → Matplotlib Visualisation
```


In [ ]:
# ============================================================
# CELL 2 — Install Java 11 + PySpark 3.5.1
# Run this cell first. Restart runtime if prompted.
# ============================================================
import subprocess, sys

def run_shell(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed:\n{result.stderr}")
    print(result.stdout.strip() or f"✔ {cmd}")

run_shell("apt-get install -y -q openjdk-11-jdk-headless")
run_shell(f"{sys.executable} -m pip install -q pyspark==3.5.1 findspark")
print("\n✅ Java + PySpark installation complete.")


In [ ]:
# ============================================================
# CELL 3 — Configure JAVA_HOME / SPARK_HOME
# ============================================================
import os, findspark

JAVA_HOME_CANDIDATES = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-openjdk-arm64",
]
JAVA_HOME = next((p for p in JAVA_HOME_CANDIDATES if os.path.isdir(p)), None)
if JAVA_HOME is None:
    raise EnvironmentError("OpenJDK 11 not found. Re-run the installation cell.")

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"]      = f"{JAVA_HOME}/bin:" + os.environ.get("PATH", "")
findspark.init()

print(f"  JAVA_HOME  : {JAVA_HOME}")
print(f"  SPARK_HOME : {os.environ.get('SPARK_HOME', 'not set')}")
print("✅ Environment configured.")


In [ ]:
# ============================================================
# CELL 4 — Core Imports
# ============================================================
import io, os, time, warnings
from pathlib import Path
from typing  import List, Tuple

import numpy as np
from PIL import Image

from pyspark.sql       import SparkSession, DataFrame
from pyspark.sql       import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, StringType
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from google.colab import files

warnings.filterwarnings("ignore")
print("✅ All imports successful.")


In [ ]:
# ============================================================
# CELL 5 — Create SparkSession
# local[*] → uses all CPU cores, mirrors multi-node cluster.
# In production: replace 'local[*]' with cluster master URL.
# ============================================================
def create_spark_session(app_name="CoastalChangeDetection"):
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.driver.memory",          "4g")
        .config("spark.executor.memory",        "4g")
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

spark = create_spark_session()
print(f"  Spark version : {spark.version}")
print(f"  Master        : {spark.sparkContext.master}")
print(f"  Cores         : {spark.sparkContext.defaultParallelism}")
print("✅ SparkSession ready.")


In [ ]:
# ============================================================
# CELL 6 — Upload Satellite Images
# Upload puthuvype_2018.png and puthuvype_2020.png.
# Missing files → synthetic data is generated automatically.
# ============================================================
EXPECTED_FILES = {"2018": "puthuvype_2018.png", "2020": "puthuvype_2020.png"}
UPLOAD_DIR     = Path("/content")

def upload_images():
    print("⬆️  Upload both images when the dialog appears …")
    try:
        uploaded = files.upload()
    except Exception:
        print("  ⚠️  Upload skipped — using synthetic data.")
        return {"2018": None, "2020": None}

    paths = {}
    for year, fname in EXPECTED_FILES.items():
        dest = UPLOAD_DIR / fname
        if fname in uploaded:
            dest.write_bytes(uploaded[fname])
            paths[year] = str(dest)
            print(f"  ✔ {fname} → {dest}")
        elif dest.exists():
            paths[year] = str(dest)
            print(f"  ✔ {fname} already present.")
        else:
            paths[year] = None
            print(f"  ⚠️  {fname} not found — synthetic data will be used.")
    return paths

image_paths = upload_images()


In [ ]:
# ============================================================
# CELL 7 — Constants + Synthetic Image Generator
# ============================================================
TARGET_SIZE  = (512, 512)    # fixed spatial resolution
TILE_SIZE    = 64            # tile edge length in pixels
N_TILES_AXIS = TARGET_SIZE[0] // TILE_SIZE  # 8 → 64 total tiles

def make_synthetic_image(year, size=TARGET_SIZE):
    """
    Generate pseudo-coastal satellite imagery.
    2020 adds deliberate change patches for realistic detection output.
    """
    rng = np.random.default_rng(seed=42 if year == "2018" else 99)
    H, W = size
    img  = np.zeros((H, W, 3), dtype=np.uint8)

    # Ocean zone (lower 40%)
    oh = int(H * 0.40)
    img[:oh, :, 0] = rng.integers( 20,  60, (oh, W))
    img[:oh, :, 1] = rng.integers( 80, 160, (oh, W))
    img[:oh, :, 2] = rng.integers(160, 220, (oh, W))

    # Beach zone (10%)
    bs, be = oh, int(H * 0.50)
    img[bs:be, :, 0] = rng.integers(180, 220, (be - bs, W))
    img[bs:be, :, 1] = rng.integers(160, 200, (be - bs, W))
    img[bs:be, :, 2] = rng.integers( 90, 140, (be - bs, W))

    # Land zone (50%)
    ls = be
    img[ls:, :, 0] = rng.integers( 50, 120, (H - ls, W))
    img[ls:, :, 1] = rng.integers( 90, 170, (H - ls, W))
    img[ls:, :, 2] = rng.integers( 30,  80, (H - ls, W))

    # Inject change patches for 2020
    if year == "2020":
        img[oh - 32 : oh + 32, 128:256]      = [10,  40,  90]   # erosion
        img[bs:be,              320:448]      = [210, 195, 120]  # accretion
        img[ls : ls + 64,        64:192]      = [130, 120, 115]  # infrastructure

    return img


def load_or_generate_image(path, year):
    if path and os.path.exists(path):
        arr = np.array(
            Image.open(path).convert("RGB").resize(TARGET_SIZE, Image.LANCZOS),
            dtype=np.uint8
        )
        print(f"  ✔ Loaded   {year} image  shape={arr.shape}")
    else:
        arr = make_synthetic_image(year)
        print(f"  ⚡ Synthetic {year} image  shape={arr.shape}")
    return arr

print("Preparing images …")
img_2018 = load_or_generate_image(image_paths.get("2018"), "2018")
img_2020 = load_or_generate_image(image_paths.get("2020"), "2020")
print("✅ Images ready.")


In [ ]:
# ============================================================
# CELL 8 — Image Tiling  (HDFS Block Partitioning Analogy)
# Each 64×64 tile ≡ one HDFS block.
# Spark will process each tile independently on a partition,
# just as HDFS TaskTrackers process blocks on local DataNodes.
# ============================================================
TileRecord = Tuple[int, int, int, float, float, float, float]

def extract_tile_features(image_array, year, tile_size=TILE_SIZE):
    """
    Partition image into tiles and extract mean RGB + mean intensity.

    Returns list of:
        (year, tile_row, tile_col, mean_r, mean_g, mean_b, mean_intensity)
    """
    H, W, _ = image_array.shape
    records  = []
    for row in range(0, H, tile_size):
        for col in range(0, W, tile_size):
            tile = image_array[row : row + tile_size, col : col + tile_size, :]
            if tile.shape[0] < tile_size or tile.shape[1] < tile_size:
                continue   # skip incomplete edge tiles
            mean_r = float(np.mean(tile[:, :, 0]))
            mean_g = float(np.mean(tile[:, :, 1]))
            mean_b = float(np.mean(tile[:, :, 2]))
            records.append((
                year,
                row // tile_size,
                col // tile_size,
                mean_r, mean_g, mean_b,
                (mean_r + mean_g + mean_b) / 3.0
            ))
    return records

print(f"Tiling images (tile size = {TILE_SIZE}×{TILE_SIZE} px) …")
t0 = time.time()
tiles_2018 = extract_tile_features(img_2018, 2018)
tiles_2020 = extract_tile_features(img_2020, 2020)
print(f"  Tiles 2018 : {len(tiles_2018)}")
print(f"  Tiles 2020 : {len(tiles_2020)}")
print(f"  Time       : {time.time() - t0:.3f}s")
print("✅ Tiling complete.")


In [ ]:
# ============================================================
# CELL 9 — Build Spark DataFrames
# parallelize() distributes tile records across Spark partitions.
# In production these records come from HDFS/S3 reads.
# ============================================================
TILE_SCHEMA = StructType([
    StructField("year",           IntegerType(), False),
    StructField("tile_row",       IntegerType(), False),
    StructField("tile_col",       IntegerType(), False),
    StructField("mean_r",         FloatType(),   False),
    StructField("mean_g",         FloatType(),   False),
    StructField("mean_b",         FloatType(),   False),
    StructField("mean_intensity", FloatType(),   False),
])

def build_spark_dataframe(records, schema, n_parts=8):
    """Parallelise tile records into a Spark DataFrame with n_parts partitions."""
    rdd = spark.sparkContext.parallelize(records, numSlices=n_parts)
    return spark.createDataFrame(rdd, schema=schema)

print("Building Spark DataFrames …")
df_2018 = build_spark_dataframe(tiles_2018, TILE_SCHEMA)
df_2020 = build_spark_dataframe(tiles_2020, TILE_SCHEMA)

print(f"  df_2018  rows={df_2018.count()}  partitions={df_2018.rdd.getNumPartitions()}")
print(f"  df_2020  rows={df_2020.count()}  partitions={df_2020.rdd.getNumPartitions()}")
print("\n— Sample rows (df_2018) —")
df_2018.show(5, truncate=False)
print("✅ Spark DataFrames ready.")


In [ ]:
# ============================================================
# CELL 10 — Distributed Change Detection
# Spark joins tile DataFrames on (tile_row, tile_col) and
# computes abs channel differences in one distributed DAG.
# Catalyst optimiser pushes predicates into each partition —
# no full data shuffle for matched partition keys.
# ============================================================
def compute_change_dataframe(df_before, df_after):
    """
    Join tile DataFrames and compute per-tile change metrics.
    Returns a cached DataFrame with delta channels + composite score.
    """
    a = df_before.alias("before")
    b = df_after .alias("after")

    return (
        a.join(b, on=["tile_row", "tile_col"], how="inner")
        .select(
            F.col("tile_row"),
            F.col("tile_col"),
            F.abs(F.col("after.mean_r")         - F.col("before.mean_r")        ).alias("delta_r"),
            F.abs(F.col("after.mean_g")         - F.col("before.mean_g")        ).alias("delta_g"),
            F.abs(F.col("after.mean_b")         - F.col("before.mean_b")        ).alias("delta_b"),
            F.abs(F.col("after.mean_intensity") - F.col("before.mean_intensity")).alias("delta_intensity"),
            F.col("before.mean_intensity").alias("intensity_2018"),
            F.col("after.mean_intensity") .alias("intensity_2020"),
        )
        .withColumn(
            "change_score",
            F.sqrt(
                F.pow(F.col("delta_r"), 2) +
                F.pow(F.col("delta_g"), 2) +
                F.pow(F.col("delta_b"), 2)
            )
        )
        .cache()   # materialise once; reused in ranking + CSV export
    )

print("Running distributed change detection …")
t0 = time.time()
df_change = compute_change_dataframe(df_2018, df_2020)
print(f"  Total tile pairs : {df_change.count()}")
print(f"  Compute time     : {time.time() - t0:.3f}s")
print("\n— Schema —")
df_change.printSchema()
print("✅ Change detection complete.")


In [ ]:
# ============================================================
# CELL 11 — Rank High-Change Areas
# ============================================================
print("— Top 15 High-Change Tiles —")
(
    df_change
    .orderBy(F.col("change_score").desc())
    .limit(15)
    .show(15, truncate=False)
)

print("— Change Score Statistics —")
df_change.select(
    F.round(F.min    ("change_score"), 2).alias("min"),
    F.round(F.max    ("change_score"), 2).alias("max"),
    F.round(F.mean   ("change_score"), 2).alias("mean"),
    F.round(F.stddev ("change_score"), 2).alias("stddev"),
    F.count("*")                         .alias("total_tiles"),
).show()


In [ ]:
# ============================================================
# CELL 12 — Export Results to CSV
# repartition(1) produces a single part file at this scale.
# Production pipelines retain multiple partitions.
# ============================================================
OUTPUT_CSV_DIR  = "/content/coastal_change_results"
OUTPUT_CSV_FILE = "/content/coastal_change_results.csv"

def save_results_csv(change_df, output_dir, output_file):
    (
        change_df
        .orderBy(F.col("change_score").desc())
        .repartition(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(output_dir)
    )
    part_files = list(Path(output_dir).glob("part-*.csv"))
    if not part_files:
        raise FileNotFoundError("No part files written — check permissions.")
    Path(output_file).write_bytes(part_files[0].read_bytes())
    size_kb = Path(output_file).stat().st_size / 1024
    print(f"  ✔ CSV written → {output_file}  ({size_kb:.1f} KB)")

save_results_csv(df_change, OUTPUT_CSV_DIR, OUTPUT_CSV_FILE)
print("✅ Results exported.")


In [ ]:
# ============================================================
# CELL 13 — Visualisation (matplotlib only)
# ============================================================
def collect_change_matrix(change_df, n_axis=N_TILES_AXIS):
    """Collect Spark results and reshape into a 2-D tile grid."""
    rows   = change_df.select("tile_row", "tile_col", "change_score").collect()
    matrix = np.zeros((n_axis, n_axis), dtype=np.float32)
    for r in rows:
        if r.tile_row < n_axis and r.tile_col < n_axis:
            matrix[r.tile_row, r.tile_col] = r.change_score
    return matrix


def plot_change_analysis(img_2018, img_2020, change_df,
                         output_path="/content/coastal_change_analysis.png"):
    """Four-panel coastal change figure."""
    change_matrix = collect_change_matrix(change_df)
    flat_scores   = change_matrix.flatten()
    tile_ids      = np.arange(len(flat_scores))
    threshold_85  = np.percentile(flat_scores, 85)

    fig = plt.figure(figsize=(18, 14), facecolor="#0d1117")
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.42, wspace=0.32,
                            left=0.05, right=0.95,
                            top=0.90,  bottom=0.08)

    T  = dict(color="white",   fontsize=12, fontweight="bold", pad=8)
    LB = dict(color="#c9d1d9", fontsize=9)

    # Panel A — 2018
    ax = fig.add_subplot(gs[0, 0])
    ax.imshow(img_2018); ax.set_title("(A)  Satellite Image — 2018", **T); ax.axis("off")

    # Panel B — 2020
    ax = fig.add_subplot(gs[0, 1])
    ax.imshow(img_2020); ax.set_title("(B)  Satellite Image — 2020", **T); ax.axis("off")

    # Panel C — Heatmap
    ax = fig.add_subplot(gs[0, 2])
    hm = ax.imshow(change_matrix, cmap="hot", interpolation="nearest",
                   vmin=0, vmax=max(change_matrix.max(), 1))
    cb = fig.colorbar(hm, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors="#c9d1d9", labelsize=8)
    cb.set_label("Change Score", color="#c9d1d9", fontsize=8)
    ax.set_title("(C)  Tile Change Heatmap", **T)
    ax.set_xlabel("Tile Column", **LB); ax.set_ylabel("Tile Row", **LB)
    ax.tick_params(colors="#c9d1d9", labelsize=8)

    # Panel D — Bar chart
    ax = fig.add_subplot(gs[1, :])
    colors = ["#e74c3c" if s >= threshold_85 else "#3498db" for s in flat_scores]
    ax.bar(tile_ids, flat_scores, color=colors, width=0.8, linewidth=0)
    ax.axhline(threshold_85, color="#f1c40f", linewidth=1.4, linestyle="--",
               label=f"85th-percentile threshold ({threshold_85:.1f})")
    ax.set_title("(D)  Tile-Wise Change Intensity  [red = high-change tiles]", **T)
    ax.set_xlabel("Tile ID (row-major order)", **LB)
    ax.set_ylabel("Change Score",              **LB)
    ax.tick_params(colors="#c9d1d9", labelsize=8)
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")
    ax.legend(facecolor="#161b22", labelcolor="#c9d1d9",
              fontsize=9, edgecolor="#30363d")

    fig.suptitle(
        "Distributed Satellite Change Detection  |  "
        "Puthuvype Beach, Kerala  |  2018 → 2020",
        color="white", fontsize=14, fontweight="bold", y=0.97
    )
    fig.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"  ✔ Figure saved → {output_path}")


print("Generating visualisation …")
plot_change_analysis(img_2018, img_2020, df_change)
print("✅ Visualisation complete.")


In [ ]:
# ============================================================
# CELL 14 — Pipeline Summary
# ============================================================
stats = df_change.agg(
    F.count("*")                                        .alias("total_tiles"),
    F.round(F.max   ("change_score"), 2)                .alias("max_change"),
    F.round(F.mean  ("change_score"), 2)                .alias("avg_change"),
    F.sum(F.when(F.col("change_score") > 50, 1)
           .otherwise(0))                               .alias("high_change_tiles"),
).collect()[0]

print("=" * 60)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 60)
print(f"  Study site         : Puthuvype Beach, Kerala, India")
print(f"  Comparison period  : 2018 → 2020")
print(f"  Image resolution   : {TARGET_SIZE[0]}×{TARGET_SIZE[1]} px")
print(f"  Tile size          : {TILE_SIZE}×{TILE_SIZE} px")
print(f"  Total tiles        : {stats.total_tiles}")
print(f"  Max change score   : {stats.max_change}")
print(f"  Avg change score   : {stats.avg_change}")
print(f"  High-change tiles  : {stats.high_change_tiles}")
print(f"  Output CSV         : {OUTPUT_CSV_FILE}")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 15 — Big Data Architecture: Interview Talking Points
# ============================================================

notes = """
╔══════════════════════════════════════════════════════════════════════╗
║          BIG DATA ARCHITECTURE — INTERVIEW TALKING POINTS          ║
╚══════════════════════════════════════════════════════════════════════╝

━━━  1.  HOW THIS SIMULATES THE HADOOP ECOSYSTEM  ━━━━━━━━━━━━━━━━━━━━

  Component          │ Hadoop Concept         │ This Project's Equivalent
  ───────────────────┼────────────────────────┼──────────────────────────
  HDFS block         │ 128 MB file chunk      │ 64×64 pixel image tile
  DataNode           │ Block storage          │ Spark executor partition
  NameNode           │ Block metadata         │ Spark DAGScheduler
  YARN ResourceMgr   │ Job scheduler          │ local[*] thread scheduler
  MapReduce Map      │ Per-block transform    │ extract_tile_features()
  MapReduce Reduce   │ Aggregation            │ Spark groupBy / join / agg
  Hive / ORC output  │ Columnar storage       │ Parquet / CSV write step

  Key principle: data is partitioned at ingest and processed where it
  lives — identical to HDFS data-locality, eliminating unnecessary I/O.

━━━  2.  WHY SPARK INSTEAD OF MAPREDUCE  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  MapReduce                         │  Spark
  ──────────────────────────────────┼───────────────────────────────────
  Disk I/O after every M/R phase    │  In-memory DAG; disk on persist()
  2-stage pipeline only             │  Arbitrary multi-stage DAG
  Java API only                     │  Python, Scala, SQL, R
  No interactive mode               │  Notebooks, REPL, Spark SQL
  10–100× slower on iterative work  │  Benchmarked faster (Apache data)

  For time-series imagery analysis, Spark caches the 2018 tile DataFrame
  in RAM and reuses it across all comparison years — something MapReduce
  cannot do without repeated HDFS reads.

━━━  3.  HOW THIS SCALES TO REAL SATELLITE BIG DATA PIPELINES  ━━━━━━━

  Step 1 — Cluster deployment
    ▸ Replace local[*] with 'yarn' or Kubernetes Spark operator
    ▸ Read scenes from HDFS / S3 via spark.read.format("image")

  Step 2 — Streaming ingestion
    ▸ Sentinel-2 / Landsat scenes via Kafka → Spark Structured Streaming
    ▸ Micro-batch tile extraction at scene arrival time

  Step 3 — Feature store
    ▸ Tile features to Delta Lake (ACID + time-travel queries)
    ▸ Reproducible change detection across arbitrary date pairs

  Step 4 — ML upgrade path
    ▸ Swap mean-intensity for CNN embeddings (TorchDistributor / MLlib)
    ▸ Cosine-similarity change score replaces abs-diff

  Production Architecture:
  ┌──────────┐  ┌──────────┐  ┌──────────────┐  ┌──────────────┐
  │Sentinel-2│→ │  Kafka   │→ │    Spark      │→ │ Delta Lake   │
  │ / GEE    │  │ (ingest) │  │  Streaming    │  │ (feat. store)│
  └──────────┘  └──────────┘  └──────┬───────┘  └──────┬───────┘
                                      │ batch            │
                               ┌──────▼───────┐  ┌──────▼───────┐
                               │ Change Det.  │→ │  Dashboard / │
                               │ (this code)  │  │  Alert Layer │
                               └──────────────┘  └──────────────┘

  Throughput estimate (10-node EMR r5.4xlarge):
    ▸ Full Kerala coastline (580 km) re-analysis, 5 years monthly
    ▸ ~70,000 tile pairs → processed in < 4 minutes end-to-end
"""

print(notes)


In [ ]:
# ============================================================
# CELL 16 — Cleanup  (run only when demo is complete)
# ============================================================
# df_change.unpersist()
# spark.stop()
# print("✅ SparkSession stopped.")
print("Pipeline complete. Uncomment the lines above to stop Spark.")
